# Key-Value Retrieval System — Medical QA Seq2Seq with Attention [Non_database_training]

## Reference
1. BLEU evaluation code adapted from [bangoc123/BLEU](https://github.com/bangoc123/BLEU).
2. Key-Value Retrieval Networks for Task-Oriented Dialogue] (https://arxiv.org/abs/1705.05414).


In [1]:
import torch

print(torch.__version__)
print(torch.version.cuda)


print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

2.12.0.dev20260407+cu128
12.8
NVIDIA GeForce RTX 5090
(12, 0)


In [2]:

import os
import random
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
import matplotlib.pyplot as plt
import sacrebleu

from typing import Tuple, List
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer

print("NumPy  version:", np.__version__)
print("Pandas version:", pd.__version__)
print("PyTorch version:", torch.__version__)

NumPy  version: 2.2.6
Pandas version: 2.3.3
PyTorch version: 2.12.0.dev20260407+cu128


/home/suman/research/key_value_retrival/key/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment set ")

Environment set 


In [4]:

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s"
)
logger = logging.getLogger(__name__)

In [5]:

def set_seed(seed: int = 42):
   random.seed(seed)
   np.random.seed(seed)
   torch.manual_seed(seed)
   torch.cuda.manual_seed_all(seed)
   torch.backends.cudnn.deterministic = True
   torch.backends.cudnn.benchmark = False

set_seed(42)

In [6]:

class Config:
    def __init__(self):
        # Model
        self.embedding_dim      = 256
        self.hidden_dim         = 256
        self.num_encoder_layers = 100
        self.num_decoder_layers = 100
        self.attention_dim      = 256
        self.dropout            = 0.4

        # Training
        self.batch_size             = 32
        self.num_epochs             = 100
        self.learning_rate          = 1e-4
        self.teacher_forcing_start  = 0.9
        self.teacher_forcing_end  = 0.2
        
        self.grad_clip              = 1.0
        self.valid_ratio            = 0.1
        self.max_seq_len            = 64

        self.grad_accum_steps   = 4

        self.model_save_path    = "/home/suman/research/key_value_retrieval/nondatabase/seq2seq_best.pth"
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = Config()
print(f"Device: {config.device}")

Device: cuda


In [7]:

print(f"CUDA available: {torch.cuda.is_available()}")
!nvidia-smi

CUDA available: True


Sat Jul  4 15:22:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        Off |   00000000:02:00.0 Off |                  N/A |
|  0%   41C    P5             24W /  575W |      18MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
df = pd.read_csv("/home/suman/research/key_value_retrieval/data/medquad.csv")
df

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
...,...,...,...,...
16402,What is (are) Diabetic Neuropathies: The Nerve...,Focal neuropathy appears suddenly and affects ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16403,How to prevent Diabetic Neuropathies: The Nerv...,The best way to prevent neuropathy is to keep ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16404,How to diagnose Diabetic Neuropathies: The Ner...,Doctors diagnose neuropathy on the basis of sy...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16405,What are the treatments for Diabetic Neuropath...,The first treatment step is to bring blood glu...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...


In [9]:
print(type(df))

<class 'pandas.core.frame.DataFrame'>


In [10]:
df = df[["question", "answer"]].reset_index(drop=True)

In [11]:
class EncoderDecoderDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = 128):
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        # SimpleTokenizer specific attributes
        self.PAD_TOKEN_ID = tokenizer.pad_token_id
        self.SOS_TOKEN_ID = tokenizer.sos_token_id
        self.EOS_TOKEN_ID = tokenizer.eos_token_id

        self.df = self._clean(df)

    def _clean(self, df):
        if "question" not in df.columns or "answer" not in df.columns:
            raise ValueError("DataFrame must have 'question' and 'answer' columns.")
        
        df = df[["question", "answer"]].copy()
        df = df.dropna().reset_index(drop=True)
        df["question"] = df["question"].astype(str).str.strip()
        df["answer"] = df["answer"].astype(str).str.strip()
        df = df[(df["question"] != "") & (df["answer"] != "")].reset_index(drop=True)
        return df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        question = self.df.loc[idx, "question"]
        answer = self.df.loc[idx, "answer"]

        # Encode
        enc_ids = self.tokenizer.encode(question, max_length=self.max_len)
        full_tgt_ids = self.tokenizer.encode(answer, max_length=self.max_len)

        # Clean PAD tokens before shifting (important!)
        full_tgt_ids = [tid for tid in full_tgt_ids if tid != self.PAD_TOKEN_ID]

        # Teacher Forcing split
        decoder_in = full_tgt_ids[:-1]      # <sos> ... token_n
        decoder_out = full_tgt_ids[1:]      # token_1 ... <eos>

        return {
            "encoder_input_ids": torch.tensor(enc_ids, dtype=torch.long),
            "decoder_input_ids": torch.tensor(decoder_in, dtype=torch.long),
            "decoder_output_ids": torch.tensor(decoder_out, dtype=torch.long),
        }

In [12]:
import logging
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

logger = logging.getLogger(__name__)


def collate_fn(batch, pad_token_id: int):
    """
    Pads variable length inputs and outputs into rectangular batch tensors.
    """
    return {
        "encoder_input_ids": pad_sequence(
            [x["encoder_input_ids"] for x in batch],
            batch_first=True, 
            padding_value=pad_token_id
        ),
        "decoder_input_ids": pad_sequence(
            [x["decoder_input_ids"] for x in batch],
            batch_first=True, 
            padding_value=pad_token_id
        ),
        # CRITICAL: -100 is automatically ignored by nn.CrossEntropyLoss
        "decoder_output_ids": pad_sequence(
            [x["decoder_output_ids"] for x in batch],
            batch_first=True, 
            padding_value=-100
        ),
    }

class CollateFn:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = pad_token_id
        
    def __call__(self, batch):
        return collate_fn(batch, self.pad_token_id)




def get_dataloader(df: pd.DataFrame, tokenizer, batch_size=32, max_len=128,
                   num_workers=0, shuffle=True) -> DataLoader:
    """
    Wraps a DataFrame into a PyTorch DataLoader utilizing highly optimized 
    memory pinning and structured collation constraints.
    """
    # Instantiate the Dataset (which handles internal cleaning)
    dataset = EncoderDecoderDataset(df, tokenizer, max_len=max_len)
    
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=CollateFn(dataset.PAD_TOKEN_ID),
        pin_memory=True,  # Speeds up tensor allocation transfers to CUDA GPUs
        drop_last=False,
        persistent_workers=False if num_workers == 0 else True, # Keeps worker threads hot
    )


def get_train_valid_dataloaders(df: pd.DataFrame, tokenizer, batch_size=32, max_len=128,
                                valid_ratio=0.1, num_workers=0, random_state=42):
    """
    Splits the clean source dataframe into training and validation sets,
    returning an optimized pair of PyTorch DataLoaders.
    """
    # --- FIXED: Streamlined Cleaning Stratum ---
    # We execute basic sanitization exactly once before the split to ensure
    # that validation ratios are computed over valid rows only.
    df = df[["question", "answer"]].dropna().reset_index(drop=True)
    df["question"] = df["question"].astype(str).str.strip()
    df["answer"] = df["answer"].astype(str).str.strip()
    df = df[(df["question"] != "") & (df["answer"] != "")].reset_index(drop=True)

    # Perform stratified/shuffled training/validation breakdown
    train_df, valid_df = train_test_split(
        df, test_size=valid_ratio, random_state=random_state, shuffle=True
    )
    
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    
    logger.info(f"Split completed: {len(train_df)} training examples / {len(valid_df)} validation examples.")

    # Generate complete DataLoader objects
    train_loader = get_dataloader(train_df, tokenizer, batch_size, max_len, num_workers, shuffle=True)
    valid_loader = get_dataloader(valid_df, tokenizer, batch_size, max_len, num_workers, shuffle=False)

    return train_loader, valid_loader

In [13]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class BiLSTMEncoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_layers,
        dropout,
        num_decoder_layers,
        pad_idx=0
    ):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_decoder_layers = num_decoder_layers
        self.pad_idx = pad_idx

        # Embedding
        self.embedding = nn.Embedding(
            vocab_size, 
            embedding_dim, 
            padding_idx=pad_idx
        )

        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)

        # Project bidirectional states to match decoder hidden size
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell = nn.Linear(hidden_dim * 2, hidden_dim)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.embedding.weight)
        
        for name, param in self.lstm.named_parameters():
            if "weight" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
                
        nn.init.xavier_uniform_(self.fc_hidden.weight)
        nn.init.zeros_(self.fc_hidden.bias)
        nn.init.xavier_uniform_(self.fc_cell.weight)
        nn.init.zeros_(self.fc_cell.bias)

    def forward(self, x):
        """
        x: (Batch, Src_Len)
        Returns:
            output:      (B, Src_Len, hidden_dim*2)   ← encoder outputs for attention
            hidden:      (num_decoder_layers, B, hidden_dim)
            cell:        (num_decoder_layers, B, hidden_dim)
            src_mask:    (B, Src_Len)
        """
        # ====================== EMBEDDING ======================
        embedded = self.dropout(self.embedding(x))                    # (B, S, E)

        # ====================== MASK ======================
        src_mask = (x != self.pad_idx).long()                         # (B, S)  ← Critical!

        # ====================== PACKED LSTM ======================
        lengths = src_mask.sum(dim=1).cpu()                           # CPU required for pack_padded_sequence
        packed = pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )

        packed_output, (hidden, cell) = self.lstm(packed)

        # ====================== UNPACK ======================
        output, _ = pad_packed_sequence(
            packed_output, batch_first=True
        )  # (B, S, 2*hidden_dim)

        # ====================== RESHAPE HIDDEN & CELL ======================
        # LSTM output shape for bidirectional: (num_layers * 2, B, hidden_dim)
        hidden = hidden.view(self.num_layers, 2, x.size(0), self.hidden_dim)
        cell = cell.view(self.num_layers, 2, x.size(0), self.hidden_dim)

        # Combine forward + backward directions
        hidden = torch.cat((hidden[:, 0], hidden[:, 1]), dim=2)   # (num_layers, B, 2*H)
        cell = torch.cat((cell[:, 0], cell[:, 1]), dim=2)         # (num_layers, B, 2*H)

        # Project to decoder dimension
        hidden = self.dropout(torch.tanh(self.fc_hidden(hidden)))
        cell = self.dropout(torch.tanh(self.fc_cell(cell)))

        # Match number of layers with Decoder
        if self.num_decoder_layers != self.num_layers:
            hidden = hidden[-1:].repeat(self.num_decoder_layers, 1, 1)
            cell = cell[-1:].repeat(self.num_decoder_layers, 1, 1)

        return output, hidden, cell, src_mask

In [14]:
class DecoderWithAttention(nn.Module):
    def __init__(self, output_dim, embed_dim, encoder_hidden_dim,
                 decoder_hidden_dim, attention_dim, num_layers=1, dropout=0.3):
        super().__init__()
        self.output_dim = output_dim
        self.decoder_hidden_dim = decoder_hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(output_dim, embed_dim, padding_idx=0)
        self.attention = AdditiveAttention(encoder_hidden_dim, decoder_hidden_dim, attention_dim)

        self.rnn = nn.LSTM(
            input_size=embed_dim + encoder_hidden_dim * 2,
            hidden_size=decoder_hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        # Combine RNN output, context, and embedding
        self.fc_out = nn.Linear(
            decoder_hidden_dim + encoder_hidden_dim * 2 + embed_dim,
            output_dim
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, decoder_hidden, decoder_cell, encoder_outputs, src_mask=None):
        """
        Args:
            input_token: (batch, 1)
            decoder_hidden: (num_layers, batch, dec_hid_dim)
            decoder_cell: (num_layers, batch, dec_hid_dim)
            encoder_outputs: (batch, src_len, enc_hid_dim*2)
            src_mask: (batch, src_len) or None. If None, all tokens are considered valid.
        """
        embedded = self.dropout(self.embedding(input_token))          # (batch, 1, embed_dim)

        # If no mask is provided, create a mask of all ones (attend to all tokens)
        if src_mask is None:
            src_mask = torch.ones(encoder_outputs.shape[:2], dtype=torch.bool, device=encoder_outputs.device)

        # Get context vector from attention (mask is now always passed)
        context, attn_weights = self.attention(encoder_outputs, decoder_hidden[-1], src_mask)
        context = context.unsqueeze(1)                                 # (batch, 1, enc_hid_dim*2)

        rnn_input = torch.cat((embedded, context), dim=2)              # (batch, 1, embed_dim + 2*H)
        output, (hidden, cell) = self.rnn(rnn_input, (decoder_hidden, decoder_cell))

        output = output.squeeze(1)          # (batch, dec_hid_dim)
        embedded = embedded.squeeze(1)      # (batch, embed_dim)
        context = context.squeeze(1)        # (batch, enc_hid_dim*2)

        combined = torch.cat((output, context, embedded), dim=1)      # (batch, total_dim)
        prediction = self.fc_out(combined)   # (batch, output_dim)
        prediction = prediction.unsqueeze(1) # (batch, 1, output_dim)

        return prediction, hidden, cell, attn_weights

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AdditiveAttention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim, attn_dim):
        super().__init__()
        self.encoder_attn = nn.Linear(enc_hid_dim * 2, attn_dim, bias=False)
        self.decoder_attn = nn.Linear(dec_hid_dim, attn_dim, bias=False)
        self.v = nn.Parameter(torch.randn(attn_dim))
        nn.init.uniform_(self.v, -0.1, 0.1)

    def forward(self, encoder_outputs, decoder_hidden, src_mask):
        """
        encoder_outputs: (B, Src_Len, enc_hid_dim*2)
        decoder_hidden:  (B, dec_hid_dim)   ← last layer hidden
        src_mask:        (B, Src_Len)
        """
        src_len = encoder_outputs.size(1)
        
        # Repeat decoder hidden for every source token
        dec_h = decoder_hidden.unsqueeze(1).repeat(1, src_len, 1)   # (B, Src_Len, dec_hid_dim)
        
        energy = torch.tanh(
            self.encoder_attn(encoder_outputs) + self.decoder_attn(dec_h)
        )
        
        scores = torch.matmul(energy, self.v)                       # (B, Src_Len)
        scores = scores.masked_fill(src_mask == 0, -1e9)           # Prevent attention on PAD
        
        weights = F.softmax(scores, dim=1)                          # (B, Src_Len)
        
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs) # (B, 1, enc_hid_dim*2)
        
        return context.squeeze(1), weights                          # (B, enc_hid_dim*2), (B, Src_Len)

In [16]:
class AdditiveAttention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim, attn_dim):
        super().__init__()
        self.encoder_attn = nn.Linear(enc_hid_dim * 2, attn_dim, bias=False)
        self.decoder_attn = nn.Linear(dec_hid_dim, attn_dim, bias=False)
        self.v = nn.Parameter(torch.randn(attn_dim))
        nn.init.uniform_(self.v, -0.1, 0.1)

    def forward(self, encoder_outputs, decoder_hidden, src_mask):
        """
        encoder_outputs: (B, Src_Len, enc_hid_dim*2)
        decoder_hidden:  (B, dec_hid_dim)
        src_mask:        (B, Src_Len)
        """
        src_len = encoder_outputs.size(1)
        
        # Ensure mask has same length as current encoder_outputs
        if src_mask.size(1) != src_len:
            src_mask = src_mask[:, :src_len]   # Truncate mask if needed
        
        dec_h = decoder_hidden.unsqueeze(1).repeat(1, src_len, 1)   # (B, Src_Len, H)
        
        energy = torch.tanh(
            self.encoder_attn(encoder_outputs) + self.decoder_attn(dec_h)
        )
        
        scores = torch.matmul(energy, self.v)                       # (B, Src_Len)
        scores = scores.masked_fill(src_mask == 0, -1e9)
        
        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs)
        
        return context.squeeze(1), weights

In [17]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        """
        src: (B, Src_Len)
        tgt: (B, Tgt_Len)  ← teacher forcing target
        """
        # ====================== ENCODER ======================
        encoder_outputs, hidden, cell, src_mask = self.encoder(src)
        
        # Safety: Ensure mask matches encoder output length (in case of any padding issues)
        if src_mask.size(1) != encoder_outputs.size(1):
            src_mask = src_mask[:, :encoder_outputs.size(1)]

        # ====================== DECODER ======================
        batch_size = tgt.size(0)
        input_token = tgt[:, 0:1]          # (B, 1)  → <SOS>
        predictions = []

        # Decode step by step
        for t in range(tgt.size(1) - 1):
            # Decoder step
            output, hidden, cell, _ = self.decoder(
                input_token, hidden, cell, encoder_outputs, src_mask
            )
            # output shape: (B, 1, vocab_size)

            predictions.append(output)

            # ====================== TEACHER FORCING ======================
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio

            if teacher_force:
                input_token = tgt[:, t+1:t+2]          # (B, 1) - ground truth
            else:
                # Greedy decoding during training
                next_token = output.argmax(dim=-1)     # (B, 1)
                input_token = next_token

        # Concatenate all predictions
        return torch.cat(predictions, dim=1)           # (B, Tgt_Len-1, vocab_size)

In [18]:
def evaluate(model, dataloader, criterion, device, pad_token_id, epoch, num_epochs):
    model.eval()
    total_loss = total_correct = total_tokens = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} [eval] ", leave=True):
            enc_ids = batch["encoder_input_ids"].to(device)
            dec_ids = batch["decoder_input_ids"].to(device)
            tgt_ids = batch["decoder_output_ids"].to(device)

            output  = model(enc_ids, dec_ids, teacher_forcing_ratio=0.0)
            targets = tgt_ids

            min_len = min(output.size(1), targets.size(1))
            output  = output[:, :min_len, :]
            targets = targets[:, :min_len]

            loss = criterion(
                output.contiguous().reshape(-1, output.size(-1)),
                targets.contiguous().reshape(-1)
            )

            mask           = targets != -100          
            total_loss    += loss.item()
            total_correct += (output.argmax(dim=-1) == targets).masked_select(mask).sum().item()
            total_tokens  += mask.sum().item()

    n_tokens = max(total_tokens, 1)                  
    return total_loss / len(dataloader), total_correct / n_tokens

In [19]:
def train_one_epoch(model, dataloader, optimizer, criterion, device,
                    pad_token_id, teacher_forcing_ratio, grad_clip,
                    epoch, num_epochs, tokenizer, grad_accum_steps=1):

    model.train()
    epoch_loss    = 0.0
    epoch_correct = 0
    epoch_tokens  = 0
    optimizer.zero_grad()

    progress = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} [train]", leave=True)

    for step, batch in enumerate(progress):
        enc_ids = batch["encoder_input_ids"].to(device)
        dec_ids = batch["decoder_input_ids"].to(device)
        tgt_ids = batch["decoder_output_ids"].to(device)

        output  = model(enc_ids, dec_ids, teacher_forcing_ratio=teacher_forcing_ratio)
        
        
        min_len = min(output.size(1), tgt_ids.size(1))
        output  = output[:, :min_len, :]
        targets = tgt_ids[:, :min_len]

        loss = criterion(
            output.reshape(-1, model.decoder.output_dim),
            targets.reshape(-1)
        )

        (loss / grad_accum_steps).backward()

        if (step + 1) % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += loss.item()
        preds  = output.argmax(dim=-1)
        mask   = targets != -100
        epoch_correct += (preds == targets).masked_select(mask).sum().item()
        epoch_tokens  += mask.sum().item()

        progress.set_postfix(loss=f"{loss.item():.4f}")

    # Flush any remaining accumulated gradients
    if len(dataloader) % grad_accum_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()
        optimizer.zero_grad()

    return (
        epoch_loss / len(dataloader),
        epoch_correct / epoch_tokens if epoch_tokens > 0 else 0.0
    )

In [20]:
@torch.no_grad()
def generate(model, tokenizer, question, device, max_len=150, temperature=0.6):
    model.eval()
    
    # Encode question
    enc_ids = tokenizer.encode(question, max_length=100)
    enc_tensor = torch.tensor([enc_ids], dtype=torch.long, device=device)
    
    encoder_outputs, hidden, cell, src_mask = model.encoder(enc_tensor)
    
    input_token = torch.tensor([[tokenizer.sos_token_id]], dtype=torch.long, device=device)
    generated = []
    
    for _ in range(max_len):
        output, hidden, cell, _ = model.decoder(
            input_token, hidden, cell, encoder_outputs, src_mask
        )
        
        logits = output.squeeze(1) / temperature
        
        # === STRONG REPETITION PENALTY ===
        if len(generated) > 4:
            for tid in generated[-15:]:
                logits[0, tid] -= 3.0   # Heavy penalty on recent tokens
        
        # Block pad
        logits[0, tokenizer.pad_token_id] = -1e9
        
        next_id = torch.argmax(logits, dim=-1).item()
        
        if next_id == tokenizer.eos_token_id:
            break
            
        generated.append(next_id)
        input_token = torch.tensor([[next_id]], dtype=torch.long, device=device)

    return tokenizer.decode(generated)

In [21]:
@torch.no_grad()
def evaluate_bleu(model, dataloader, tokenizer, device, max_batches=50):
    """
    Compute BLEU score using SimpleTokenizer
    """
    model.eval()
    hypotheses = []
    references = []
    
    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, desc="Evaluating BLEU")):
            if i >= max_batches:
                break
                
            enc_ids = batch["encoder_input_ids"].to(device)
            dec_ids = batch["decoder_input_ids"].to(device)
            tgt_ids = batch["decoder_output_ids"]
            
            # Forward pass with no teacher forcing
            output = model(enc_ids, dec_ids, teacher_forcing_ratio=0.0)
            pred_ids = output.argmax(dim=-1).cpu().tolist()
            
            tgt_list = tgt_ids.tolist()
            
            for pred, ref in zip(pred_ids, tgt_list):
                # Skip special tokens
                skip_tokens = {tokenizer.pad_token_id, tokenizer.eos_token_id}
                
                # Clean prediction
                pred_clean = [t for t in pred if t not in skip_tokens]
                
                # Clean reference
                ref_clean = [t for t in ref if t not in skip_tokens and t != -100]
                
                hypotheses.append(tokenizer.decode(pred_clean))
                references.append(tokenizer.decode(ref_clean))
    
    # Compute BLEU
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    print(f"BLEU-4 Score: {bleu.score:.2f}")
    print(f"BLEU-1: {bleu.precisions[0]:.2f} | BLEU-2: {bleu.precisions[1]:.2f} | BLEU-3: {bleu.precisions[2]:.2f}")
    
    return bleu.score

In [22]:
import torch
from collections import Counter
import pandas as pd

class SimpleTokenizer:
    def __init__(self, texts, min_freq=2):
        self.PAD = "<pad>"
        self.SOS = "<sos>"
        self.EOS = "<eos>"
        self.UNK = "<unk>"

        self.special_tokens = [self.PAD, self.SOS, self.EOS, self.UNK]
        self.word2idx = {}
        self.idx2word = {}

        self.build_vocab(texts, min_freq)

    def tokenize(self, text):
        if not isinstance(text, str):
            text = str(text)
        return text.lower().strip().split()

    def build_vocab(self, texts, min_freq):
        counter = Counter()
        for text in texts:
            counter.update(self.tokenize(text))

        # Special tokens first
        for tok in self.special_tokens:
            self.word2idx[tok] = len(self.word2idx)

        # Regular tokens
        for word, freq in counter.items():
            if freq >= min_freq:
                self.word2idx[word] = len(self.word2idx)

        self.idx2word = {v: k for k, v in self.word2idx.items()}

    def encode(self, text, max_length=None):
        if not isinstance(text, str):
            text = str(text)
        tokens = self.tokenize(text)
        ids = [self.word2idx.get(t, self.word2idx[self.UNK]) for t in tokens]
        
        encoded = [self.word2idx[self.SOS]] + ids + [self.word2idx[self.EOS]]
        
        if max_length:
            if len(encoded) > max_length:
                encoded = encoded[:max_length-1] + [self.word2idx[self.EOS]]
            else:
                encoded = encoded + [self.word2idx[self.PAD]] * (max_length - len(encoded))
        return encoded

    def decode(self, ids, skip_special_tokens=True):
        words = []
        for i in ids:
            if i not in self.word2idx.values():
                continue
            w = self.idx2word.get(i, self.UNK)
            if skip_special_tokens and w in self.special_tokens:
                continue
            words.append(w)
        return " ".join(words)

    @property
    def vocab_size(self):
        return len(self.word2idx)

    @property
    def pad_token_id(self):
        return self.word2idx[self.PAD]

    @property
    def sos_token_id(self):
        return self.word2idx[self.SOS]

    @property
    def eos_token_id(self):
        return self.word2idx[self.EOS]




In [ ]:
def main():
    # ====================== DATA CLEANING ======================
    df_clean = df[["question", "answer"]].copy()
    df_clean = df_clean.dropna().reset_index(drop=True)
    df_clean["question"] = df_clean["question"].astype(str).str.strip()
    df_clean["answer"] = df_clean["answer"].astype(str).str.strip()
    df_clean = df_clean[(df_clean["question"] != "") & (df_clean["answer"] != "")].reset_index(drop=True)

    print(f"Cleaned dataset size: {len(df_clean)} rows")

    # ====================== SimpleTokenizer ======================
    all_texts = df_clean["question"].tolist() + df_clean["answer"].tolist()
    tokenizer = SimpleTokenizer(texts=all_texts, min_freq=2)

    print(f" Vocabulary size: {tokenizer.vocab_size:,}")
    print(f" PAD: {tokenizer.pad_token_id} | SOS: {tokenizer.sos_token_id} | EOS: {tokenizer.eos_token_id}")

    # ====================== DATALOADERS ======================
    train_loader, valid_loader = get_train_valid_dataloaders(
        df_clean, tokenizer,                    
        batch_size=config.batch_size, 
        max_len=config.max_seq_len,
        valid_ratio=config.valid_ratio, 
        num_workers=0, 
        random_state=42,
    )

    # ====================== MODEL ======================
    vocab_size = tokenizer.vocab_size

    encoder = BiLSTMEncoder(
        vocab_size=vocab_size, 
        embedding_dim=config.embedding_dim,
        hidden_dim=config.hidden_dim, 
        num_layers=config.num_encoder_layers,
        dropout=config.dropout, 
        num_decoder_layers=config.num_decoder_layers,
        pad_idx=tokenizer.pad_token_id          
    )

    decoder = DecoderWithAttention(
        output_dim=vocab_size, 
        embed_dim=config.embedding_dim,
        encoder_hidden_dim=config.hidden_dim, 
        decoder_hidden_dim=config.hidden_dim,
        attention_dim=config.attention_dim, 
        num_layers=config.num_decoder_layers,
        dropout=config.dropout,
    )

    model = Seq2Seq(encoder, decoder).to(config.device)
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # ====================== TRAINING SETUP ======================
    pad_token_id = tokenizer.pad_token_id
    criterion = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.2)
    optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=1e-5)    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2)

    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    best_val_loss = float("inf")

    import os
    os.makedirs(os.path.dirname(config.model_save_path), exist_ok=True)

    # ====================== TRAINING LOOP ======================
    for epoch in range(config.num_epochs):
        print(f"\n{'='*60}")
        print(f" Epoch {epoch+1}/{config.num_epochs} | LR: {optimizer.param_groups[0]['lr']:.2e}")
        print(f"{'='*60}")

        denom = max(config.num_epochs - 1, 1)
        tf_ratio = config.teacher_forcing_start - (
            (config.teacher_forcing_start - config.teacher_forcing_end) * (epoch / denom)
        )

        t_loss, t_acc = train_one_epoch(
            model, train_loader, optimizer, criterion,
            device=config.device, 
            pad_token_id=pad_token_id,
            teacher_forcing_ratio=tf_ratio,
            grad_clip=config.grad_clip, 
            epoch=epoch,
            tokenizer=tokenizer,
            num_epochs=config.num_epochs, 
            grad_accum_steps=config.grad_accum_steps,
        )

        v_loss, v_acc = evaluate(
            model, valid_loader, criterion,
            device=config.device, 
            pad_token_id=pad_token_id,
            epoch=epoch, 
            num_epochs=config.num_epochs,
        )

        scheduler.step(v_loss)

        train_losses.append(t_loss)
        train_accs.append(t_acc)
        val_losses.append(v_loss)
        val_accs.append(v_acc)

        print(f" Train → loss: {t_loss:.4f} | acc: {t_acc:.4f}")
        print(f" Valid → loss: {v_loss:.4f} | acc: {v_acc:.4f}")

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save(model.state_dict(), config.model_save_path)
            print(f" ✓ Best model saved (val_loss={v_loss:.4f})")

    
    
    
    # ====================== FINAL EVALUATION ======================
    print("\n" + "="*70)
    print("FINAL EVALUATION & GENERATION")
    print("="*70)
    
    evaluate_bleu(model, valid_loader, tokenizer, config.device)

    # ====================== GENERATION TEST ======================
    print("\n" + "="*70)
    print("SAMPLE GENERATIONS")
    print("="*70)

    test_questions = [
        "What is (are) Glaucoma ?",
        "What causes Glaucoma ?",
        "What are the symptoms of Diabetes ?",
        "How to treat Hypertension ?",
        "What is the treatment for COVID-19 ?"
    ]

    for q in test_questions:
        print(f"\n🔹 Question: {q}")
        ans = generate(model, tokenizer, q, config.device, max_len=120, temperature=0.65)
        print(f" Answer  : {ans}")
        print("-" * 80)
    # ====================== PLOT TRAINING CURVES ======================
    print("Plotting training curves...")
    epochs = range(1, config.num_epochs + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
   
    axes[0].plot(epochs, train_losses, label="Train Loss", marker="o")
    axes[0].plot(epochs, val_losses,   label="Val Loss",   marker="o")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(epochs, train_accs, label="Train Acc", marker="o")
    axes[1].plot(epochs, val_accs,   label="Val Acc",   marker="o")
    axes[1].set_title("Token Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig("/home/suman/research/key_value_retrieval/nondatabase/training_curves.png", dpi=150)
    plt.show()

: 

In [ ]:
if __name__ == "__main__":
    try:
        mp.set_start_method("spawn", force=False)
    except RuntimeError: 

        
        pass
    main()

Cleaned dataset size: 16407 rows


2026-07-04 15:22:56,627 — INFO — Split completed: 14766 training examples / 1641 validation examples.


 Vocabulary size: 38,984
 PAD: 0 | SOS: 1 | EOS: 2
Trainable params: 270,707,528

 Epoch 1/100 | LR: 1.00e-04


Epoch 1/100 [train]:  49%|████▉     | 227/462 [09:52<10:16,  2.62s/it, loss=7.6635] 